# 04 - Stopping-rule zoo (decision gate G2)
Every post-hoc rule is evaluated at every threshold in one pass over each test set (thresholds
chosen on ID-val, `src/glwm2/readouts/rules.py`); the jointly trained gates (ACT, PonderNet,
joint correctness head, v1 GLWM gate) come from their own runs. Checks **H1** (length leakage of
pooled gates) and **H2** (premature stopping of confidence/convergence rules), then **G2**.

In [ ]:
# parameters
CONFIGS = ["configs/main_addition.yaml", "configs/main_parity.yaml", "configs/main_gates.yaml"]
TASKS = ["addition", "parity"]
SPLITS = ["id_test", "ood_depth", "ood_length", "ood_both"]

In [ ]:
%matplotlib inline
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from glwm2 import PROJECT_ROOT, RESULTS_DIR, PAPER_DIR
from glwm2.eval.plotting import set_style
set_style()
WRITTEN = []  # files this notebook writes

In [ ]:
from glwm2.eval.results import configs_from_yaml, finished, curve_rows, step_acc
from glwm2.eval.analysis import (ENTRIES, BASELINES, OURS, POOLED_JOINT, CONF_RULES, entry_rows, entry_instances,
                                 summary_table, g2_check, g3_check, binned, premature_ci, length_leakage_ci)
from glwm2.eval.plotting import savefig, FAMILY_COLORS
from glwm2.utils.io import write_json
cfgs = {t: [c for f in CONFIGS for c in configs_from_yaml(PROJECT_ROOT / f) if c.task == t] for t in TASKS}
for t in TASKS:
    print(t, f"{len(finished(cfgs[t]))}/{len(cfgs[t])} runs finished, seeds:",
          sorted({c.seed for c in finished(cfgs[t])}))
rows = {t: entry_rows(cfgs[t]) for t in TASKS}

In [ ]:
for t in TASKS:
    print(f"== {t}: accuracy (mean over seeds)")
    display(summary_table(rows[t], SPLITS, "acc")["mean"].round(3))

In [ ]:
for t in TASKS:
    print(f"== {t}: premature-stop rate and mean steps on OOD-depth / OOD-both")
    sub = rows[t][rows[t].split.isin(["ood_depth", "ood_both"])]
    display(sub.pivot_table(index="entry", columns="split", values=["premature", "mean_steps", "mean_lb"], aggfunc="mean")
            .reindex([e for e in ENTRIES if e in set(sub.entry)]).round(3))

In [ ]:
# H1: length leakage (partial Spearman of steps with length given depth) on ID-test + OOD-length,
# and whether pooled gates add steps when depth exceeds training (OOD-depth mean steps vs mean lower bound)
h1, g2 = {}, {}
for t in TASKS:
    inst_len = pd.concat([entry_instances(cfgs[t], s) for s in ["id_test", "ood_length"]])
    inst_deep = pd.concat([entry_instances(cfgs[t], s) for s in ["ood_depth", "ood_both"]])
    res = g2_check(inst_deep, inst_len)
    od = rows[t][rows[t].split == "ood_depth"].groupby("entry")[["mean_steps", "mean_lb"]].mean()
    res = res.merge(od, left_on="entry", right_index=True, how="left")
    h1[t] = res; display(res.round(3))
    g2[t] = bool(res.shows_premature.any() or res.shows_leak.any())
print("G2 (some baseline family shows premature stopping or length leakage, CI excluding 0):", g2)

In [ ]:
# Fig. 4: steps against depth (OOD-depth + ID) and against length (ID + OOD-length), addition
show = ["oracle_lb", "max_softmax", "stability", "kl", "act", "ponder", "v1gate", "trm_joint", "done_prop_post", "done_joint"]
fig, axes = plt.subplots(len(TASKS), 2, figsize=(7.0, 2.4 * len(TASKS)), squeeze=False)
for i, t in enumerate(TASKS):
    deep = pd.concat([entry_instances(cfgs[t], s, show) for s in ["id_test", "ood_depth", "ood_both"]])
    longi = pd.concat([entry_instances(cfgs[t], s, show) for s in ["id_test", "ood_length"]])
    for ax, inst, col, bins in [(axes[i, 0], deep, "depth", [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 32, 48, 64, 96, 128]),
                                (axes[i, 1], longi, "length", [4, 8, 12, 16, 20, 64, 96, 128, 192, 256])]:
        g = binned(inst, col, bins)
        for e in show:
            ge = g[g.entry == e]
            if ge.empty: continue
            fam = ENTRIES[e][2]
            ax.plot(ge.x, ge.steps, marker=".", lw=1.2 if fam == "ours" else 0.8, ls="-" if fam != "reference" else ":",
                    color=FAMILY_COLORS[fam], alpha=1.0 if fam in ("ours", "reference") else 0.6, label=ENTRIES[e][3])
        ax.set_xscale("log", base=2); ax.set_yscale("log", base=2)
        ax.set_xlabel("depth $d^*$" if col == "depth" else "length L"); ax.set_ylabel("steps used"); ax.set_title(t)
axes[0, 0].legend(fontsize=5.5, frameon=False, ncol=2)
WRITTEN.append(savefig(fig, "fig4_steps_vs_depth_length"))

In [ ]:
# Fig. 5: accuracy-versus-compute curves on OOD-depth (every threshold of every rule)
fig, axes = plt.subplots(1, len(TASKS), figsize=(7.0, 2.6), squeeze=False)
for ax, t in zip(axes[0], TASKS):
    cur = curve_rows(finished(cfgs[t]))
    cur = cur[cur.split == "ood_depth"]
    for e, (method, rule, fam, label) in ENTRIES.items():
        sub = cur[(cur.method == method) & (cur.rule == rule)]
        if sub.empty or e in ("oracle_iter",): continue
        g = sub.groupby("threshold", dropna=False)[["acc", "mean_steps"]].mean().sort_values("mean_steps")
        ax.plot(g.mean_steps, g.acc, marker="." if len(g) > 1 else "*", ms=4 if len(g) > 1 else 8,
                lw=1.2 if fam == "ours" else 0.7, color=FAMILY_COLORS[fam], alpha=0.9 if fam != "posthoc" else 0.6,
                label=label if fam in ("ours", "reference") or e in ("max_softmax", "act", "ponder") else None)
    ax.set_xlabel("mean steps"); ax.set_ylabel("accuracy"); ax.set_title(f"{t}, OOD-depth"); ax.set_xscale("log")
axes[0, 0].legend(fontsize=5.5, frameon=False)
WRITTEN.append(savefig(fig, "fig5_tradeoff"))

In [ ]:
# Fig. 6: premature-stop rate against depth
fig, axes = plt.subplots(1, len(TASKS), figsize=(7.0, 2.4), squeeze=False)
for ax, t in zip(axes[0], TASKS):
    inst = pd.concat([entry_instances(cfgs[t], s, show) for s in ["id_test", "ood_depth", "ood_both"]])
    inst["premature"] = (inst.steps < inst.lb).astype(float)
    g = binned(inst, "depth", [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 32, 48, 64, 96, 128], value="premature")
    for e in show:
        ge = g[g.entry == e]
        if ge.empty or e == "oracle_lb": continue
        fam = ENTRIES[e][2]
        ax.plot(ge.x, ge.premature, marker=".", color=FAMILY_COLORS[fam], lw=1.2 if fam == "ours" else 0.7, label=ENTRIES[e][3])
    ax.axvline(10, color="grey", ls=":", lw=0.8)
    ax.set_xscale("log", base=2); ax.set_xlabel("depth $d^*$"); ax.set_ylabel("premature-stop rate"); ax.set_title(t)
axes[0, 0].legend(fontsize=5.5, frameon=False, ncol=2)
WRITTEN.append(savefig(fig, "fig6_premature_vs_depth"))

In [ ]:
WRITTEN.append(write_json(RESULTS_DIR / "gates" / "g2.json", {"G2": g2, **{t: h1[t].to_dict(orient="records") for t in TASKS}}))

In [ ]:
print("Files written by this notebook:")
for p in WRITTEN:
    print(" ", Path(p).relative_to(PROJECT_ROOT))